# Lección 11 - Protocolo Agente a Agente (A2A)


## Configuración


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## Creación de Agentes de Viaje Especializados


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Colaboración Multi-Agente a través de Flujo de Trabajo

Conectamos los tres agentes en un flujo de trabajo secuencial que refleja el paso de mensajes A2A:

1. **CurrencyExchangeAgent** recibe la solicitud del usuario y produce orientación sobre divisas.
2. **ActivityPlannerAgent** recibe el contexto enriquecido y añade recomendaciones de actividades.
3. **TravelManagerAgent** sintetiza ambas entradas en un informe final de viaje.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Entendiendo A2A en Producción

En un entorno de producción, el protocolo A2A desbloquea potentes escenarios entre servicios:

| Capacidad | Descripción |
|---|---|
| **Interoperabilidad entre frameworks** | Un agente construido con un framework puede delegar tareas a un agente construido con cualquier otro framework compatible con A2A, permitiendo una verdadera interoperabilidad entre organizaciones. |
| **Límites de servicio** | Los agentes pueden vivir en microservicios separados, regiones de la nube o incluso en diferentes organizaciones, mientras colaboran sin problemas. |
| **Descubrimiento dinámico** | Un orquestador puede consultar un registro de Agent Card en tiempo de ejecución para encontrar el especialista más adecuado para una sub-tarea dada. |
| **Transmisión y notificaciones push** | A2A soporta Server-Sent Events (SSE) para actualizaciones en tiempo real del progreso y notificaciones push para tareas de larga duración. |

El flujo de trabajo que construimos arriba es una versión simplificada y en proceso de este patrón. En un despliegue real
cada agente expondría un endpoint HTTP, publicaría una Agent Card y comunicaría
vía el protocolo JSON-RPC de A2A.


## Resumen

En esta lección aprendiste:

1. **Qué es el protocolo A2A** — un estándar abierto para el descubrimiento, la mensajería
   y la gestión de tareas entre agentes.
2. **Cómo crear agentes especializados** — un agente de intercambio de divisas, un agente planificador de actividades,
   y un orquestador de gestor de viajes.
3. **Cómo conectar agentes en un flujo de trabajo** — usando `WorkflowBuilder` para modelar el paso secuencial
   de mensajes entre agentes.
4. **Cómo funciona A2A en producción** — permitiendo colaboración entre diferentes frameworks y servicios
   con descubrimiento dinámico y actualizaciones en streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
